In [0]:
from pyspark.sql.functions import when, col , try_to_date
customer_df = spark.read.csv("/Volumes/vivek/practise/vkalivasan/customers.csv", header=True, inferSchema=True)
products_df = spark.read.csv("/Volumes/vivek/practise/vkalivasan/products.csv", header=True, inferSchema=True)
orders_df = spark.read.csv("/Volumes/vivek/practise/vkalivasan/orders.csv", header=True, inferSchema=True)
display(customer_df)
customer_df = customer_df.withColumn("city", when(col("city") == "NA", "UNKNOWN").otherwise(col("city")))
display(customer_df)
customer_df1 = customer_df.dropDuplicates(["customer_id"])
display(customer_df1)
orders_df1 = orders_df.withColumn("order_date", try_to_date(col("order_date"), "M/d/yyyy"))
display(orders_df1)
orders_df1.printSchema()
silver_df = orders_df1.join(products_df, "product_id").join(customer_df1, "customer_id")
display(silver_df)
silver_df1 = silver_df.withColumn("final_amount", col("price")*col("quantity"))
display(silver_df1)
count_df = silver_df.filter(col("name") == "Vivek")
count_df.count()

In [0]:
customer_df = spark.read.csv("/Volumes/vivek/practise/vkalivasan/customers.csv", header=True, inferSchema=True)
products_df = spark.read.csv("/Volumes/vivek/practise/vkalivasan/products.csv", header=True, inferSchema=True)
orders_df = spark.read.csv("/Volumes/vivek/practise/vkalivasan/orders.csv", header=True, inferSchema=True)
customer_df.join(orders_df,"customer_id","inner").show()
customer_df.join(orders_df,"customer_id","left").show()
customer_df.join(orders_df,"customer_id","left_anti").show()
customer_df.join(orders_df,"customer_id","left_semi").show()
customer_df.join(orders_df,"customer_id","right").show()
customer_df.join(orders_df,"customer_id","full").show()

In [0]:
from pyspark.sql.functions import when, col , try_to_date, sum, dense_rank
from pyspark.sql.window import Window
silver_df1.show()
citywise_revenue_df = silver_df1.groupBy("city").agg(sum("final_amount").alias("total_revenue"))
citywise_revenue_df.write.mode("overwrite").format("delta").save("/Volumes/vivek/practise/vkalivasan/citywise_revenue")
window_spec = Window.orderBy(col("total_revenue").desc())
citywise_rank_df = citywise_revenue_df.withColumn("rank", dense_rank().over(window_spec))
display(citywise_rank_df)